# 세포 인스턴스 분할 평가 결과

기존 학습 실험의 집계값과 평가 설정을 로드한다. 기록 시점은 JSON의 `recorded_at`을 따른다. 수록된 수치는 공개 모듈을 이용한 전체 재학습 결과와 구분한다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

In [ ]:
import json
from IPython.display import display, Markdown
record = json.loads((ROOT/'results/historical-summary.json').read_text())
print('Recorded at', record['recorded_at'])
print('Test images', record['test_images'], '| precision', record['precision'])

In [ ]:
lines = ['| Model | AP@0.5 | AP@0.75 | Mean image ms |', '|---|---:|---:|---:|']
for model in record['models']:
    lines.append(f"| {model['model']} | {model['ap50']:.4f} | {model['ap75']:.4f} | {model['mean_image_ms']:.1f} |")
display(Markdown('\n'.join(lines)))
print('ViT-S snapshot', record['vits'])

In [ ]:
baseline = record['models'][0]
for model in record['models'][1:]:
    print(model['model'], {'observed_latency_ratio_vs_L':round(baseline['mean_image_ms']/model['mean_image_ms'],3),
        'AP50_delta':round(model['ap50']-baseline['ap50'],5)})
print(*record['limitations'], sep='\n')

AP는 이미지별 Cellpose instance matching 점수의 평균이다. 지연시간은 이미지당 3회 측정 중앙값의 평균이며 디스크 읽기와 모델 로딩은 제외한다. 지도학습 ViT-S는 500 epochs와 64장 test 평가를 완료하였다. 추가 증류 상태와 결과는 아래 별도 기록으로 구분한다. 반복 실험 분산과 GPU 동시 작업 통제 기록이 없으므로 관측된 차이의 통계적 유의성은 평가할 수 없다.

In [ ]:
distillation = json.loads((ROOT/'results/vits-distillation.json').read_text())
print('ViT-S additional distillation', distillation)
if distillation['status'] == 'complete':
    print({key:distillation[key] for key in ['ap50','ap75','count_mae','mean_image_ms']})